# Converting LBM Spartan Data to Tar Shards

This notebook walks through downloading raw LBM Spartan data and converting it to WebDataset tar shards for VLA Foundry training.

We use `PutGreenAppleOnSaucer` and `PickAndPlaceBox` as examples.

## Prerequisites

- Python 3.12+ with VLA Foundry via `uv`
- **`Python (vla_foundry)`** kernel:
  ```bash
  bash tutorials/install_kernel.sh
  ```
- Restart kernel after installing
- Disk space:
  - **~35 GB network download** (raw tars for `PickAndPlaceBox` + `PutGreenAppleOnSaucer`).
  - **~75 GB peak on-disk** (raw tars + extracted `tasks/` tree + preprocessed WebDataset shards).
  - You can delete the raw `.tar` files after extraction to reclaim ~35 GB.


In [ ]:
import os
import subprocess

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
repo_root = subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip()
os.chdir(repo_root)
print(f"Working directory: {os.getcwd()}")

**Switch to Python (vla_foundry) kernel and restart before continuing.**

## Available LBM Spartan Tasks

The following tasks are available in the LBM Spartan dataset:

| Task | Size |
|------|------|
| BimanualHangMugsOnMugHolderFromDryingRack | 92.4 GB |
| BimanualHangMugsOnMugHolderFromTable | 93.3 GB |
| BimanualLayCerealBoxOnCuttingBoardFromTopShelf | 53.2 GB |
| BimanualLayCerealBoxOnCuttingBoardFromUnderShelf | 80.6 GB |
| BimanualPlaceAppleFromBowlIntoBin | 76.1 GB |
| BimanualPlaceAppleFromBowlOnCuttingBoard | 73.0 GB |
| BimanualPlaceAvocadoFromBowlOnCuttingBoard | 113.1 GB |
| BimanualPlaceFruitFromBowlIntoBin | 164.6 GB |
| BimanualPlaceFruitFromBowlOnCuttingBoard | 140.3 GB |
| BimanualPlacePearFromBowlIntoBin | 75.6 GB |
| BimanualPlacePearFromBowlOnCuttingBoard | 113.4 GB |
| BimanualPutMugsOnPlatesFromDryingRack | 135.3 GB |
| BimanualPutMugsOnPlatesFromTable | 69.6 GB |
| BimanualPutRedBellPepperInBin | 70.5 GB |
| BimanualPutSpatulaOnPlateFromDryingRack | 50.6 GB |
| BimanualPutSpatulaOnPlateFromTable | 38.5 GB |
| BimanualPutSpatulaOnTableFromDryingRack | 61.9 GB |
| BimanualPutSpatulaOnTableFromUtensilCrock | 88.5 GB |
| BimanualStackPlatesOnTableFromDryingRack | 149.9 GB |
| BimanualStackPlatesOnTableFromTable | 215.8 GB |
| BimanualStoreCerealBoxUnderShelf | 74.9 GB |
| PickAndPlaceBox | 15.6 GB |
| PlaceCupByCoaster | 121.7 GB |
| PlaceCupOnCoaster | 199.3 GB |
| PushCoasterToCenterOfTable | 164.4 GB |
| PushCoasterToMug | 182.8 GB |
| PutBananaInCenterOfTable | 20.2 GB |
| PutBananaOnSaucer | 23.3 GB |
| PutCupInCenterOfTable | 95.8 GB |
| PutCupOnSaucer | 201.2 GB |
| PutGreenAppleInCenterOfTable | 55.9 GB |
| PutGreenAppleOnSaucer | 19.9 GB |
| PutKiwiInCenterOfTable | 19.6 GB |
| PutKiwiOnSaucer | 43.0 GB |
| PutMugOnSaucer | 115.6 GB |
| PutOrangeInCenterOfTable | 44.8 GB |
| PutOrangeOnSaucer | 20.9 GB |
| PutSpatulaInUtensilCrock | 54.2 GB |
| PutSpatulaInUtensilCrockFromDryingRack | 46.2 GB |
| TurnCupUpsideDown | 394.2 GB |
| TurnMugRightsideUp | 220.3 GB |

**Total: 40 tasks, ~3.8 TB**. Raw data is not available for PushBox — use the pre-processed download instead.

## Step 1: Download Raw Spartan Data

Download raw episodes using `download_dataset.py` with the `--raw` flag.

> **Heads up on download + disk.** The next cell downloads raw Spartan data for two tasks (`PickAndPlaceBox` 15.6 GB + `PutGreenAppleOnSaucer` 19.9 GB) — **~35 GB over the network**. The cell is idempotent (skips tars already on disk with the matching size; pass `--force` to re-download).
>
> At 50 MB/s this takes ~12 minutes; under `jupyter nbconvert` the default 30 s per-cell timeout will trip. Either run interactively, pass `--ExecutePreprocessor.timeout=3600`, or run the download from a terminal first.

In [ ]:
from tutorials.tutorial_utils import download_lbm_spartan_raw

LOCAL_RAW_PATH = "./tutorials/data/lbm_raw"

download_lbm_spartan_raw(
    tasks=["PutGreenAppleOnSaucer", "PickAndPlaceBox"],
    local_path=LOCAL_RAW_PATH,
)

## Step 2: Find diffusion_spartan Directories

Locate all `diffusion_spartan/` directories in the downloaded data.

In [ ]:
import glob

spartan_dirs = glob.glob(f"{LOCAL_RAW_PATH}/tasks/**/diffusion_spartan/", recursive=True)
spartan_dirs = [os.path.abspath(d) for d in spartan_dirs]

print(f"Found {len(spartan_dirs)} directories:")
for d in spartan_dirs:
    print(f"  {d}")

## Step 3: Preprocess to Tar Shards

Convert the raw Spartan data to WebDataset format using LBM-specific configs.

In [ ]:
import json

OUTPUT_DIR = os.path.abspath("./tutorials/data/preprocessed/lbm_mixed_tasks/")
os.environ["UV_PROJECT_ENVIRONMENT"] = os.path.abspath(".venv")

source_episodes_json = json.dumps(spartan_dirs)

!uv run --group preprocessing python vla_foundry/data/preprocessing/preprocess_robotics_to_tar.py \
  --type "spartan" \
  --source_episodes '{source_episodes_json}' \
  --output_dir {OUTPUT_DIR} \
  --camera_names "include vla_foundry/config_presets/data/lbm/lbm_data_camera_names_4cameras.yaml" \
  --language_annotations_path vla_foundry/config_presets/data/lbm/lbm_language_annotations.yaml \
  --action_fields_config_path vla_foundry/config_presets/data/lbm/lbm_action_fields.yaml \
  --data_discard_keys "include vla_foundry/config_presets/data/lbm/lbm_data_discard_key.yaml" \
  --samples_per_shard 100 \
  --config_path "vla_foundry/config_presets/data/robotics_preprocessing_params_1past_14future.yaml"